In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path
from ast import literal_eval

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.tools import aggregate_results

In [3]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def highlight_extrema(s, use_italics=False):  
    styles = []
    for v in s:      
        style = ''
        if not isinstance(v, (int, float, np.floating)):
            pass
        elif v == s.max():
            style = 'font-weight: bold;'
        elif v == s.min():
            style = 'font-weight: bold; font-style: italic;'
        styles.append(style)
    
    return styles

def sig3(x):
    if isinstance(x, (int, float, np.floating)):
        return f"{x:.3g}"
    return x

In [13]:
### Load F1 data

metric_cols_f1 = ["theme f1 mean", "topic f1 mean", "concept f1 mean"]

res = pd.read_csv("./data/metrics_v3.csv", sep=";")
res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])
res["suite"] = res.apply(get_suite, axis=1)

f1s = res[["model", "suite"] + metric_cols_f1]

f1s = f1s.rename(
    columns={
        "model": "Model",
        "suite": "Suite",
        "theme f1 mean": "Theme",
        "topic f1 mean": "Topic",
        "concept f1 mean": "Concept",
    }
)

f1s["Sum"] = f1s[f1s.columns[-3:]].sum(axis=1)

In [14]:
### Baseline, max, min for each model and label
# In Use!
models = np.sort(f1s["Model"].unique())
metric_cols = list(f1s.columns[-4:])

baselines = f1s["Suite"] == "0-non-expl"

max_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmax()
min_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmin()

best_per_label = pd.concat(
    [
        f1s[baselines],
        f1s.loc[max_values.values.reshape(-1)],
        f1s.loc[min_values.values.reshape(-1)],
    ]
).groupby(["Model", "Suite"]).max()

for model in models:
    per_model = best_per_label.loc[model]

    latex = per_model.style.apply(
        lambda col: highlight_extrema(
            col,
            True,
        )
    ).format(sig3).to_latex(convert_css=True, column_format="lllll")
    
    print("%" + model)
    print(latex)

%Llama-70B
\begin{tabular}{lllll}
 & Theme & Topic & Concept & Sum \\
Suite &  &  &  &  \\
0-non-expl & 0.939 & \bfseries 0.977 & 0.858 & 2.77 \\
1-neg-expl & \bfseries \itshape 0.894 & \bfseries \itshape 0.929 & 0.831 & 2.65 \\
1-pos-impl & 0.919 & 0.951 & \bfseries \itshape 0.435 & \bfseries \itshape 2.3 \\
6-pos-expl & \bfseries 0.943 & 0.971 & \bfseries 0.875 & \bfseries 2.79 \\
\end{tabular}

%Llama-8B
\begin{tabular}{lllll}
 & Theme & Topic & Concept & Sum \\
Suite &  &  &  &  \\
0-non-expl & 0.697 & 0.83 & 0.749 & 2.28 \\
1-pos-impl & \bfseries \itshape 0.444 & 0.588 & \bfseries \itshape 0.635 & \bfseries \itshape 1.67 \\
6-mix-expl & \bfseries 0.868 & 0.858 & \bfseries 0.902 & \bfseries 2.63 \\
6-pos-expl & 0.806 & \bfseries 0.906 & 0.887 & 2.6 \\
6-pos-impl & 0.478 & \bfseries \itshape 0.567 & 0.7 & 1.75 \\
\end{tabular}

%Mistral-24B
\begin{tabular}{lllll}
 & Theme & Topic & Concept & Sum \\
Suite &  &  &  &  \\
0-non-expl & 0.852 & 0.888 & 0.663 & 2.4 \\
1-neg-impl & 0.691 &

In [15]:
### Table with models ordered by their best score per label
# In Use!

top_scores = f1s.loc[max_values.values.reshape(-1)].groupby(["Model"]).max()

models_ordered_by_scores = {}

for i, col in enumerate(metric_cols):
    best_scores_for_col = top_scores.loc[:, col].reset_index()

    # Order models by F1 scores
    ordered = best_scores_for_col.sort_values(by=col, ascending=False)

    # Add place for mean diff
    models_ordered = ordered["Model"].to_list()
    models_ordered.append("Mean")
    models_ordered_by_scores[(col, "Model")] = models_ordered

    # Get diffs to top score
    diffs = (ordered[col] - ordered[col].max()).to_list()
    diffs[0] = ordered[col].max() 
    diffs.append(np.mean(diffs[1:]))

    models_ordered_by_scores[(col, "F1 (Top, $\\Delta$)")] = diffs


ordered_df = pd.DataFrame(models_ordered_by_scores)
latex = ordered_df.to_latex(index=False, float_format="%.3g", column_format="lr|lr|lr|lr")

print(latex)

\begin{tabular}{lr|lr|lr|lr}
\toprule
\multicolumn{2}{r}{Theme} & \multicolumn{2}{r}{Topic} & \multicolumn{2}{r}{Concept} & \multicolumn{2}{r}{Sum} \\
Model & F1 (Top, $\Delta$) & Model & F1 (Top, $\Delta$) & Model & F1 (Top, $\Delta$) & Model & F1 (Top, $\Delta$) \\
\midrule
Llama-70B & 0.943 & Llama-70B & 0.977 & Llama-8B & 0.902 & Llama-70B & 2.79 \\
Qwen3-32B & -0.00463 & Qwen3-32B & -0.00847 & Llama-70B & -0.0275 & Llama-8B & -0.16 \\
Mistral-24B & -0.0533 & Mistral-24B & -0.0642 & Qwen3-32B & -0.159 & Qwen3-32B & -0.228 \\
Llama-8B & -0.0748 & Llama-8B & -0.0709 & Mistral-24B & -0.178 & Mistral-24B & -0.282 \\
Qwen3-8B & -0.149 & Qwen3-8B & -0.105 & Mistral-7B & -0.21 & Qwen3-8B & -0.66 \\
Mistral-7B & -0.432 & Mistral-7B & -0.272 & Qwen3-8B & -0.226 & Mistral-7B & -0.922 \\
Mean & -0.143 & Mean & -0.104 & Mean & -0.16 & Mean & -0.45 \\
\bottomrule
\end{tabular}



In [17]:
### Same as above but for baselines

bl = f1s.loc[baselines].set_index("Model")

models_ordered_by_scores = {}

for i, col in enumerate(metric_cols):
    best_scores_for_col = bl.loc[:, col].reset_index()

    # Order models by F1 scores
    ordered = best_scores_for_col.sort_values(by=col, ascending=False)

    # Add place for mean diff
    models_ordered = ordered["Model"].to_list()
    models_ordered.append("Mean")
    models_ordered_by_scores[(col, "Model")] = models_ordered

    # Get diffs to top score
    diffs = (ordered[col] - ordered[col].max()).to_list()
    diffs[0] = ordered[col].max() 
    diffs.append(np.mean(diffs[1:]))

    models_ordered_by_scores[(col, "F1 (Top, $\\Delta$)")] = diffs


ordered_df = pd.DataFrame(models_ordered_by_scores)
latex = ordered_df.to_latex(index=False, float_format="%.3g", column_format="lr|lr|lr|lr")

print(latex)

\begin{tabular}{lr|lr|lr|lr}
\toprule
\multicolumn{2}{r}{Theme} & \multicolumn{2}{r}{Topic} & \multicolumn{2}{r}{Concept} & \multicolumn{2}{r}{Sum} \\
Model & F1 (Top, $\Delta$) & Model & F1 (Top, $\Delta$) & Model & F1 (Top, $\Delta$) & Model & F1 (Top, $\Delta$) \\
\midrule
Llama-70B & 0.939 & Llama-70B & 0.977 & Llama-70B & 0.858 & Llama-70B & 2.77 \\
Qwen3-32B & -0.0132 & Qwen3-32B & -0.0131 & Llama-8B & -0.109 & Qwen3-32B & -0.315 \\
Mistral-24B & -0.0875 & Mistral-24B & -0.0892 & Mistral-24B & -0.195 & Mistral-24B & -0.372 \\
Qwen3-8B & -0.154 & Qwen3-8B & -0.105 & Qwen3-32B & -0.288 & Llama-8B & -0.498 \\
Llama-8B & -0.242 & Llama-8B & -0.147 & Qwen3-8B & -0.447 & Qwen3-8B & -0.706 \\
Mistral-7B & -0.507 & Mistral-7B & -0.357 & Mistral-7B & -0.804 & Mistral-7B & -1.67 \\
Mean & -0.201 & Mean & -0.142 & Mean & -0.369 & Mean & -0.712 \\
\bottomrule
\end{tabular}

